In [1]:
# ----------------------------
# script to create batch
# files for SMEW simulations
# ----------------------------

import os
import itertools
import numpy as np
import pandas as pd
import pickle
from scipy.stats import qmc
import warnings

import create_batch_fxns as bfx

## Parameter dict options

There are a few types of acceptable parameter dictionaries.

1. Parameter ranges for latin hypercube sampling --- each parameter has [min, max] in that order
```
parameter_ranges = {
    "b_area": [21000, 50000000],
    "basedep": [100, 3e3],
    "f": [0.01, 0.99],
    "iceflag": [".false.", ".true."],
}
```

2. Parameter values to generate all combinations of the input values
```
parameter_values = {
    "b_area": [21000, 210000, 2100000, 21000000],
    "basedep": [100, 500, 1.5e3, 3e3],
    "f": [0.01, 0.25, 0.5, 0.75, 0.99],
    "iceflag": [".false.", ".true."],
}
```

3. Parameter cases where each key has values of the same lengh, with each index referring to a single simulation
```
parameter_cases = {
    "b_area": [21000, 210000, 2100000, 21000000],
    "basedep": [100, 500, 1.5e3, 3e3],
    "f": [0.01, 0.25, 0.75, 0.99],
    "iceflag": [".false.", ".true.", ".false.", ".true."],
}
```

4. Parameter one-at-a-time --- lists for individual parameter sensitivity tests
```
parameter_one_at_a_time = {
    "b_area": [21000, 210000, 2100000, 21000000],
    "basedep": [100, 500, 1.5e3, 3e3],
    "f": [0.01, 0.25, 0.75, 0.99],
    "iceflag": [".false."],
}
```

In [2]:
# --- name the batch file 
BATCHNAME = 'site+tstep+mineral_v0'
savehere = '/home/tykukla/EWmodel/batch_inputs'
# --- create the parameter dictionary 
# ... NOTE: each 'mineral' has to be a list of names (even if it's a single mineral)
parameter_dict = {
    'sitename': ["albany", "atlanta", "minneapolis", "central_valley"],
    'climfn': ["hourly.nc", "daily.nc", "monthly.nc", "yearly.nc"],
    'feedstock_name': ['wollastonite', 'forsterite'],
    'M_rock_in': [0, 2000], 
}


## apply the desired function
Examples:
1. Latin Hypercube
```
latin_hypercube_sampler(parameter_ranges, n_samples=100, 
                                 nonnum_repeat_type="prescribed_cases")
```
2. all combinations
```
all_combinations_sampler(parameter_values)
```
3. prescribed cases
```
prescribed_cases(parameter_cases)
```
4. one at a time
```
one_at_a_time_sensitivity(parameter_one_at_a_time)
```

In [3]:
# --- apply the desired function
df = bfx.all_combinations_sampler(parameter_dict)

# --- add control label
df['control'] = np.where(df['M_rock_in'] == 0, "Y", "N")
df

,sitename,climfn,feedstock_name,M_rock_in,control
0,albany,hourly.nc,wollastonite,0,Y
1,albany,hourly.nc,wollastonite,2000,N
2,albany,hourly.nc,forsterite,0,Y
3,albany,hourly.nc,forsterite,2000,N
4,albany,daily.nc,wollastonite,0,Y
...,...,...,...,...,...
59,central_valley,monthly.nc,forsterite,2000,N
60,central_valley,yearly.nc,wollastonite,0,Y
61,central_valley,yearly.nc,wollastonite,2000,N
62,central_valley,yearly.nc,forsterite,0,Y


## Add constant values

In [4]:
constant_dict = {
    # --- THE ONLY TWO REQUIRED COLUMNS ---
    # [UPDATE TO PATH ON YOUR MACHINE]
    "default_name": "default_main_inputClim",
    'outdir': "s3://carbonplan-carbon-removal/SMEW/smew_output/",
    # --------------------------------------
    # 
    # other constants to override defaults
    # ...
    'SOC_perc': 1.0,  # [%] percent SOC in
    'CEC_tot': 13,    # [mmol_c per 100g dry soil] cation exchange capacity
    'pH_in': 5.8,     # [] initial pH of soil water
}

dfout = bfx.add_constant_parameters(df, constant_dict)

## Create final dictionary with custom run naming rule

In [6]:
batch_dict = {
    f"{row['sitename']}-{row['climfn'].replace('.nc', '')}-{row['feedstock_name'][0]}-rckflx_{row['M_rock_in']}": row.to_dict()
    for _, row in dfout.iterrows()
}

# make sure no runs were lost (imprecise naming can lead to overwrites)
if len(batch_dict) != len(dfout):
    print("YIKES! runs were lost, choose a new naming convention")
else:
    print("All runs accounted for :)")

All runs accounted for :)


In [7]:
list(batch_dict.values())[0]
# list(batch_dict.keys())[0]

{'sitename': 'albany',
 'climfn': 'hourly.nc',
 'feedstock_name': 'wollastonite',
 'M_rock_in': 0,
 'control': 'Y',
 'default_name': 'default_main_inputClim',
 'outdir': 's3://carbonplan-carbon-removal/SMEW/smew_output/',
 'SOC_perc': 1.0,
 'CEC_tot': 13,
 'pH_in': 5.8}

In [8]:
# --- save the result
with open(os.path.join(savehere, f'{BATCHNAME}.pkl'), 'wb') as f:
    pickle.dump(batch_dict, f)

In [83]:
# ---